# Qwen2.5-Math-1.5B Evaluation with MoTTT
This notebook evaluates the baseline `Qwen/Qwen2.5-Math-1.5B-Instruct` and compares it to `Qwen/Qwen2.5-Math-1.5B` enhanced with the MoTTT (Test-Time LoRA Scratchpad) method.

In [ ]:
!git clone https://github.com/Jerryliu3547/MoTTT.git /content/MoTTT
%cd /content/MoTTT
!pip install -q -e .


In [ ]:
import os
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from mottt.models.mottt_model import MoTTTModel


## 1. Evaluate Baseline (Qwen2.5-Math-1.5B-Instruct)

In [ ]:
baseline_name = "Qwen/Qwen2.5-Math-1.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Loading {baseline_name}...")
baseline_tokenizer = AutoTokenizer.from_pretrained(baseline_name, trust_remote_code=True)
baseline_model = AutoModelForCausalLM.from_pretrained(
    baseline_name,
    torch_dtype=torch_dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)

baseline_pipe = pipeline(
    "text-generation",
    model=baseline_model,
    tokenizer=baseline_tokenizer,
    max_new_tokens=512,
    do_sample=False
)


In [ ]:
import json
import re
from tqdm import tqdm

def extract_predicted_answer(text: str) -> str:
    if "####" in text:
        after_hash = text.split("####")[-1].strip()
        cleaned = re.sub(r"[,\$]", "", after_hash).strip()
        tokens = cleaned.split()
        if tokens:
            return tokens[0].strip()
    match = re.search(r"(?:the\s+answer\s+is\s+|is\s+|equal\s+to\s+)([-+]?\d+(?:\.\d+)?)", text, re.IGNORECASE)
    if match:
        return match.group(1).replace(",", "").strip()
    numbers = re.findall(r"[-+]?\d+(?:\.\d+)?", text)
    if numbers:
        return numbers[-1].replace(",", "").strip()
    return ""

from typing import List, Dict, Any
from pathlib import Path
from mottt.data.dataset_exporter import load_distractor_jsonl

def find_or_load_dataset(split: str, preferred_paths: List[Path]) -> List[Dict[str, Any]]:
    for p in preferred_paths:
        if p.exists():
            records = load_distractor_jsonl(str(p))
            print(f"Loaded {len(records)} {split} records from: {p}")
            return records

    print(f"Notice: No local {split} dataset found in candidate paths. Generating synthetic {split} records...")
    depth_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]
    synthetic_records = []
    base_math_problems = [
        {
            "query": "Janet sells remaining duck eggs at the farmers' market daily for $2 each. If she has 9 duck eggs left, how much does she make?",
            "premise": "Janet has 9 fresh duck eggs left to sell at the market for $2 each.",
            "gold_answer": "18",
            "solution": "She has 9 eggs. Each sells for $2. 9 * 2 = 18. #### 18",
        },
        {
            "query": "James decides to run 3 miles every day for 2 weeks. How many miles does he run altogether?",
            "premise": "James runs 3 miles every day. 2 weeks has 14 days.",
            "gold_answer": "42",
            "solution": "2 weeks has 14 days. 14 * 3 = 42. #### 42",
        },
    ]
    for prob_idx, p in enumerate(base_math_problems):
        for depth in depth_ratios:
            distractor_prefix = "In an ancient historical chronicle, scholars studied various ancient artifacts and botanical archives. " * int(depth * 10 + 1)
            distractor_suffix = "The botanical manuscripts describe historical flora and regional geographical topography in great detail. " * int((1.0 - depth) * 10 + 1)
            full_context = distractor_prefix + "\n[ARCHIVE NOTE: " + p["premise"] + "]\n" + distractor_suffix
            rec = {
                "id": f"{split}_synth_{prob_idx}_depth_{int(depth*100)}",
                "query": p["query"],
                "premise": p["premise"],
                "distractor_context": full_context,
                "needle_depth_ratio": depth,
                "gold_answer": p["gold_answer"],
                "solution": p["solution"],
            }
            synthetic_records.append(rec)
    return synthetic_records

test_candidates = [
    Path("/content/drive/MyDrive/MoTTT/data/gsm8k_distractor_test/gsm8k_distractor_all.jsonl"),
    Path("/content/gsm8k_distractor_all.jsonl"),
]

all_records = find_or_load_dataset("test", test_candidates)
records = all_records[:100]
print(f"Using {len(records)} test records.")
correct = 0
    
    for rec in tqdm(records, desc="Evaluating Baseline"):
        ctx = rec.get('distractor_context', '')[:1500]
        query = rec.get('query', '')
        gold = str(rec.get('gold_answer', '')).strip()
        original_question = rec.get('original_question', query)
        
        prompt = f"Background Context:\n{ctx}\n\nQuestion:\n{query}\n\nPlease solve the problem step by step and end your response with '#### [final numerical answer]'."
        outputs = baseline_pipe(prompt)
        gen_text = outputs[0]["generated_text"][len(prompt):].strip()
        pred = extract_predicted_answer(gen_text)
        
        tqdm.write(f"\n{'-'*60}")
        tqdm.write(f"Question: {original_question}")
        tqdm.write(f"Gold:     {gold}")
        tqdm.write(f"Pred:     {pred}  (Matches: {pred == gold})")
        tqdm.write(f"Model Gen:\n{gen_text}")
        tqdm.write(f"{'-'*60}")
        
        if pred == gold:
            correct += 1
            
    print(f"\nBaseline Accuracy on First {len(records)} Samples: {correct/len(records)*100:.1f}% ({correct}/{len(records)})")


## 2. Evaluate MoTTT Model (Qwen2.5-Math-1.5B + Query-Aware Router)

In [ ]:
# Note: In a real scenario, you'd load Qwen/Qwen2.5-Math-1.5B base model, 
# and load trained mottt_router.pt and reasoning_experts.pt from your checkpoint directory.

base_name = "Qwen/Qwen2.5-Math-1.5B"
print(f"Loading {base_name} for MoTTT...")
mottt_tokenizer = AutoTokenizer.from_pretrained(base_name, trust_remote_code=True)
hf_backbone = AutoModelForCausalLM.from_pretrained(
    base_name,
    torch_dtype=torch_dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)

mottt_model = MoTTTModel(
    hidden_dim=hf_backbone.config.hidden_size,
    num_reasoning_experts=4, # Assuming default config
    rank=16,
    alpha=16.0,
    base_backbone=hf_backbone,
    all_linear=True
).to(device)

# Example: Load your trained checkpoints here
# ckpt_dir = Path("../gsm8k/checkpoints")
# mottt_model.router.load_state_dict(torch.load(ckpt_dir / "mottt_router.pt"))
# Add loading logic for reasoning_experts.pt here


In [ ]:
# Create pipeline with adapted backbone
mottt_pipe = pipeline(
    "text-generation",
    model=hf_backbone,
    tokenizer=mottt_tokenizer,
    max_new_tokens=512,
    do_sample=False
)

if 'records' in locals() and len(records) > 0:
    correct = 0
    inner_steps = 1
    inner_opt = torch.optim.SGD(mottt_model.get_scratchpad_parameters(), lr=1e-3)
    
    for rec in tqdm(records, desc="Evaluating MoTTT"):
        ctx = rec.get('distractor_context', '')[:1500]
        query = rec.get('query', '')
        gold = str(rec.get('gold_answer', '')).strip()
        
        # 1. Inner Loop Adaptation
        mottt_model.reset_scratchpad()
        mottt_model.set_routing_gates(None)
        
        ctx_enc = mottt_tokenizer(ctx, truncation=True, max_length=256, return_tensors="pt").to(device)
        for _ in range(inner_steps):
            ctx_out = hf_backbone(input_ids=ctx_enc.input_ids, attention_mask=ctx_enc.attention_mask)
            shift_logits = ctx_out.logits[..., :-1, :].contiguous()
            shift_labels = ctx_enc.input_ids[..., 1:].contiguous()
            inner_loss = F.cross_entropy(shift_logits.view(-1, hf_backbone.config.vocab_size), shift_labels.view(-1))
            inner_opt.zero_grad()
            inner_loss.backward()
            inner_opt.step()
            
        # 2. Query-Aware Routing
        q_enc = mottt_tokenizer(query, return_tensors="pt").to(device)
        with torch.no_grad():
            q_emb = hf_backbone.model.embed_tokens(q_enc.input_ids).mean(dim=1)
            gates, _ = mottt_model.router(q_emb.unsqueeze(0).unsqueeze(0), q_emb.unsqueeze(0))
        mottt_model.set_routing_gates(gates)
        
        # 3. Generation
        prompt = f"Background Context:\n{ctx}\n\nQuestion:\n{query}\n\nPlease solve the problem step by step and end your response with '#### [final numerical answer]'."
        outputs = mottt_pipe(prompt)
        gen_text = outputs[0]["generated_text"][len(prompt):].strip()
        pred = extract_predicted_answer(gen_text)
        
        original_question = rec.get('original_question', query)
        tqdm.write(f"\n{'-'*60}")
        tqdm.write(f"Question: {original_question}")
        tqdm.write(f"Gold:     {gold}")
        tqdm.write(f"Pred:     {pred}  (Matches: {pred == gold})")
        tqdm.write(f"Model Gen:\n{gen_text}")
        tqdm.write(f"{'-'*60}")
        
        if pred == gold:
            correct += 1
            
        mottt_model.reset_scratchpad()
        
    print(f"\nMoTTT Accuracy on First {len(records)} Samples: {correct/len(records)*100:.1f}% ({correct}/{len(records)})")
else:
    print("Please load the test records first.")
